In [10]:
import pandas as pd
import numpy as np
import re, ast, pickle, torch, os
import torch.nn as nn
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [11]:
df = pd.read_csv('../Data/news.tsv', sep='\t', on_bad_lines='skip')

df["Headline"] = df["Headline"].astype(str)
df["Title entity"] = df["Title entity"].astype(str)

BIO CONVERSION

In [12]:
COUNTRIES = [
    "United States", "India", "Brazil", "China", "Mexico", "Canada", "United Kingdom", "Germany", "France", "Italy", "Spain",
    "Netherlands", "Sweden", "Norway", "Denmark", "Switzerland", "Belgium", "Austria","Japan", "South Korea", "North Korea", 
    "Indonesia", "Thailand", "Malaysia", "Singapore", "Vietnam", "Philippines", "Pakistan","Bangladesh", "Sri Lanka", "Nepal", 
    "Saudi Arabia", "United Arab Emirates", "Qatar", "Kuwait", "Oman", "Israel", "Iran", "Turkey", "South Africa", "Nigeria", 
    "Kenya", "Egypt", "Morocco", "Ethiopia", "Argentina", "Chile", "Colombia", "Peru", "Venezuela", "Australia", "New Zealand"
]

PERSON_PATTERN = r"^[A-Z][a-z]+(\s[A-Z][a-z]+)+$"

ORG_KEYWORDS = [
    "Company", "Ltd", "Limited", "Inc", "Incorporated", "LLC", "Group", "Corporation", "Authority", "Committee", "Association", 
    "University", "Enterprises", "Industries", "Solutions", "Technologies", "Systems", "Ministry", "Department", "Agency", 
    "Council", "Commission", "Board", "Parliament", "Senate", "Government", "Administration", "College", "Institute", "School", 
    "Academy", "Research Center", "Laboratory", "Lab", "Organization", "Foundation", "Fund", "Trust", "Mission", "Union", 
    "Alliance", "Forum", "Network", "Club", "FC", "Team", "League", "Federation", "Media", "News", "Press", "Studio", "Channel",
    "Bank", "Finance", "Capital", "Holdings", "Investments"
]

def infer_entity_type(expanded):
    if re.match(PERSON_PATTERN, expanded):
        return "PERSON"
    if expanded.lower() in [c.lower() for c in COUNTRIES]:
        return "LOCATION"
    if any(k in expanded for k in ORG_KEYWORDS):
        return "ORG"
    return "MISC"

def convert_to_bio(text, entity_string):
    tokens = text.split()
    tags = ["O"] * len(tokens)

    try:
        ent_dict = ast.literal_eval(entity_string)
    except:
        ent_dict = {}

    lower_tokens = [w.lower() for w in tokens]

    for surface, expanded in ent_dict.items():
        stoks = surface.lower().split()
        n = len(stoks)
        ent_type = infer_entity_type(expanded)

        for i in range(len(tokens)):
            if lower_tokens[i:i+n] == stoks:
                tags[i] = f"B-{ent_type}"
                for j in range(i+1, i+n):
                    if j < len(tags):
                        tags[j] = f"I-{ent_type}"

    for i in range(len(tokens) - 1):
        full_name = tokens[i] + " " + tokens[i+1]

        if re.match(PERSON_PATTERN, full_name):
            
            if tags[i] == "O" and tags[i+1] == "O":
                tags[i] = "B-PERSON"
                tags[i+1] = "I-PERSON"

    return tokens, tags

In [13]:
sentences, labels = [], []

for _, row in df.iterrows():
    s, t = convert_to_bio(row["Headline"], row["Title entity"])
    sentences.append(s)
    labels.append(t)

VOCAB & ENCODE

In [14]:
word_set = set(w for s in sentences for w in s)
tag_set = set(t for seq in labels for t in seq)

word2idx = {w: i+2 for i, w in enumerate(word_set)}
word2idx["<PAD>"] = 0
word2idx["<UNK>"] = 1

tag2idx = {t: i for i, t in enumerate(tag_set)}
idx2tag = {v: k for k, v in tag2idx.items()}

MAX_LEN = 60

def encode_data(sentences, labels):
    X = [[word2idx.get(w, 1) for w in s] for s in sentences]
    y = [[tag2idx[t] for t in l] for l in labels]

    X = [x[:MAX_LEN] + [0]*(MAX_LEN-len(x)) for x in X]
    y = [t[:MAX_LEN] + [0]*(MAX_LEN-len(t)) for t in y]

    return np.array(X), np.array(y)

X, y = encode_data(sentences, labels)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

Glove

In [15]:
GLOVE_PATH = "../Data/glove.6B.100d.txt"

embedding_matrix = np.random.normal(size=(len(word2idx), 100))

with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        if word in word2idx:
            embedding_matrix[word2idx[word]] = np.asarray(values[1:], dtype="float32")

BiLSTM + Glove

In [16]:
import tensorflow as tf
from tensorflow.keras.layers import *

def build_bilstm():
    inp = Input(shape=(MAX_LEN,))
    emb = Embedding(len(word2idx), 100, weights=[embedding_matrix], trainable=False)(inp)
    x = Bidirectional(LSTM(128, return_sequences=True))(emb)
    out = TimeDistributed(Dense(len(tag2idx), activation="softmax"))(x)

    model = tf.keras.Model(inp, out)
    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
    return model

bilstm_model = build_bilstm()
bilstm_model.fit(X_train, y_train, epochs=5, batch_size=32)

Epoch 1/5
2845/2845 ━━━━━━━━━━━━━━━━━━━━ 53s 17ms/step - loss: 0.1037
Epoch 2/5
2845/2845 ━━━━━━━━━━━━━━━━━━━━ 48s 17ms/step - loss: 0.0749
Epoch 3/5
2845/2845 ━━━━━━━━━━━━━━━━━━━━ 48s 17ms/step - loss: 0.0638
Epoch 4/5
2845/2845 ━━━━━━━━━━━━━━━━━━━━ 48s 17ms/step - loss: 0.0562
Epoch 5/5
2845/2845 ━━━━━━━━━━━━━━━━━━━━ 48s 17ms/step - loss: 0.0504


BiLSTM + CRF + GLOVE

In [17]:
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float),
            freeze=False
        )

        self.lstm = nn.LSTM(100, 128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, len(tag2idx))
        self.crf = CRF(len(tag2idx), batch_first=True)

    def forward(self, input_ids, tags=None):

        mask = (input_ids != 0)

        x = self.embedding(input_ids)
        x, _ = self.lstm(x)
        emissions = self.fc(x)

        if tags is not None:
            return -self.crf(emissions, tags, mask=mask)
        else:
            return self.crf.decode(emissions, mask=mask)

In [18]:
from tqdm import tqdm

model_crf = BiLSTM_CRF().to(device)
optimizer = torch.optim.Adam(model_crf.parameters(), lr=0.001)

EPOCHS = 5
BATCH_SIZE = 32

for epoch in range(EPOCHS):
    model_crf.train()
    total_loss = 0

    print(f"\n Epoch {epoch+1}/{EPOCHS} started...")

    loop = tqdm(range(0, len(X_train), BATCH_SIZE), desc="Training", leave=False)

    for i in loop:
        xb = torch.tensor(X_train[i:i+BATCH_SIZE], dtype=torch.long).to(device)
        yb = torch.tensor(y_train[i:i+BATCH_SIZE], dtype=torch.long).to(device)

        loss = model_crf(xb, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        loop.set_postfix({
            "Batch Loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / (len(X_train) // BATCH_SIZE)

    print(f" Epoch {epoch+1}/{EPOCHS} completed | Avg Loss: {avg_loss:.4f}")


 Epoch 1/5 started...


 Epoch 1/5 completed | Avg Loss: 127.0992

 Epoch 2/5 started...


 Epoch 2/5 completed | Avg Loss: 62.0404

 Epoch 3/5 started...


 Epoch 3/5 completed | Avg Loss: 41.2846

 Epoch 4/5 started...


 Epoch 4/5 completed | Avg Loss: 27.6520

 Epoch 5/5 started...


 Epoch 5/5 completed | Avg Loss: 17.5352


Evaluation

In [19]:
from tqdm import tqdm

def predict_crf_in_batches(model, X, batch_size=8):
    model.eval()
    all_preds = []

    with torch.no_grad():
        for i in tqdm(range(0, len(X), batch_size), desc="Evaluating CRF"):
            xb = torch.tensor(X[i:i+batch_size], dtype=torch.long).to(device)

            preds = model(xb)

            all_preds.extend(preds)

    return all_preds

In [20]:
def evaluate_model(preds, true):

    y_true, y_pred = [], []

    for t_seq, p_seq in zip(true, preds):
        for t, p in zip(t_seq, p_seq):
            if idx2tag[t] != "O":
                y_true.append(idx2tag[t])
                y_pred.append(idx2tag[p])

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    result = {
        "Precision": report["weighted avg"]["precision"],
        "Recall": report["weighted avg"]["recall"],
        "F1": report["weighted avg"]["f1-score"],
        "Micro F1": report.get("micro avg", {}).get("f1-score", 0)
    }

    for label in report:
        if label not in ["accuracy", "macro avg", "weighted avg", "micro avg"]:
            result[f"{label}_F1"] = report[label]["f1-score"]

    return result

In [21]:
pred_bilstm = bilstm_model.predict(X_val).argmax(-1)
rep1 = evaluate_model(pred_bilstm, y_val)

pred_crf = predict_crf_in_batches(model_crf, X_val)
rep2 = evaluate_model(pred_crf, y_val)

rep1["Model"] = "BiLSTM + Glove"
rep2["Model"] = "BiLSTM + CRF + Glove"

df_res = pd.DataFrame([rep1, rep2])
df_res.to_csv("NER_comparison.csv", index=False)

print("\nRESULTS:\n", df_res)

712/712 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step


Evaluating CRF: 100%|██████████| 2845/2845 [00:21<00:00, 130.18it/s]



RESULTS:
    Precision    Recall        F1  Micro F1  B-LOCATION_F1  B-MISC_F1  \
0   0.857681  0.694335  0.760835         0       0.793143   0.559030   
1   0.889892  0.790729  0.834635         0       0.847608   0.673926   

   B-ORG_F1  B-PERSON_F1  I-LOCATION_F1  I-MISC_F1  I-ORG_F1  I-PERSON_F1  \
0   0.37247     0.795600            0.0   0.511905  0.538071     0.801789   
1   0.50805     0.865332            0.0   0.609923  0.596330     0.865912   

   O_F1                 Model  
0   0.0        BiLSTM + Glove  
1   0.0  BiLSTM + CRF + Glove  


In [22]:
best = df_res.loc[df_res["F1"].idxmax()]
print("\nBEST MODEL:\n", best)


BEST MODEL:
 Precision                    0.889892
Recall                       0.790729
F1                           0.834635
Micro F1                            0
B-LOCATION_F1                0.847608
B-MISC_F1                    0.673926
B-ORG_F1                      0.50805
B-PERSON_F1                  0.865332
I-LOCATION_F1                     0.0
I-MISC_F1                    0.609923
I-ORG_F1                      0.59633
I-PERSON_F1                  0.865912
O_F1                              0.0
Model            BiLSTM + CRF + Glove
Name: 1, dtype: object


In [23]:
ARTIFACTS_DIR = "./artifacts/DL"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

with open(os.path.join(ARTIFACTS_DIR, "word2idx.pkl"), "wb") as f:
    pickle.dump(word2idx, f)

with open(os.path.join(ARTIFACTS_DIR, "idx2tag.pkl"), "wb") as f:
    pickle.dump(idx2tag, f)

if "CRF" in best["Model"]:
    torch.save(model_crf.state_dict(), os.path.join(ARTIFACTS_DIR, "best_model_crf.pt"))
else:
    bilstm_model.save(os.path.join(ARTIFACTS_DIR, "best_model.keras"))